In [1]:
import time

import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
import os
print("cwd:", os.getcwd())
print("files here:", os.listdir("."))
print("data/ here:", os.listdir("data") if os.path.isdir("data") else "MISSING")

kr = "\n".join(line for line in open("data/kr.txt", encoding="utf-8").read().splitlines() if line.strip())
eng = "\n".join(line for line in open("data/eng.txt", encoding="utf-8").read().splitlines() if line.strip())
chn = "\n".join(line for line in open("data/chn.txt", encoding="utf-8").read().splitlines() if line.strip())

kr_tokens = list(map(int, kr.encode('utf-8')))
en_tokens = list(map(int, eng.encode('utf-8')))
ch_tokens = list(map(int, chn.encode('utf-8')))
tokens = kr_tokens + en_tokens + ch_tokens
print(f"len(kr_tokens):{len(kr_tokens)}", kr_tokens[: 10])
print(f"len(en_tokens):{len(en_tokens)}", en_tokens[: 10])
print(f"len(ch_tokens):{len(ch_tokens)}", ch_tokens[: 10])
print(f"len(tokens):{len(tokens)}", list(tokens)[: 1000])


cwd: /content
files here: ['.config', 'sample_data']
data/ here: MISSING


FileNotFoundError: [Errno 2] No such file or directory: 'data/kr.txt'

In [ ]:
from collections import Counter, defaultdict

def get_ngram_stats(ids, min_n=2, max_n=5):
    if min_n < 2 or max_n < 2 or min_n >= max_n:
        raise ValueError("min_n and max_n must be greater than or equal to 2.")

    stats = defaultdict(int)
    size = len(ids)
    for n in range(min_n, max_n + 1):
        for i in range(size - n + 1):
            ngram = tuple(ids[i:i+n])
            stats[ngram] += 1
    return stats

def get_status(ids):
    stats = {}
    # for i in range(len(ids) - 1):
    #     a, b = ids[i], ids[i + 1]
    #     pair = (a, b)
    #     stats[pair] = stats.get(pair, 0) + 1
    for a, b in zip(ids, ids[1:]):
        pair = (a, b)
        stats[pair] = stats.get(pair, 0) + 1
    return stats

status = get_status(tokens)
sorted_status = sorted(((v, k) for (k, v) in status.items()), reverse=True)
print(f"len(sorted_status):{len(sorted_status)}", list(sorted_status)[: 10])
print("most common bigrams:", [chr(a) + chr(b) for (_, (a, b)) in sorted_status[: 10]])
chr(32) + chr(236)

len(sorted_status):4784 [(4923, (32, 236)), (3212, (236, 157)), (2508, (32, 235)), (2160, (101, 32)), (1758, (237, 149)), (1727, (115, 32)), (1705, (32, 97)), (1672, (105, 110)), (1621, (32, 116)), (1403, (44, 32))]
most common bigrams: [' ì', 'ì\x9d', ' ë', 'e ', 'í\x95', 's ', ' a', 'in', ' t', ', ']


' ì'

In [ ]:
def get_stats(ids):
    return Counter(zip(ids, ids[1:]))

def merge_once(ids, pair, new_token):
    a, b = pair
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_token)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

def train_bpe(ids, start_token=256, threshold=5):
    bpe_map = {}  # new_token -> (a, b)
    new_token = start_token
    ids = list(ids)

    while True:
        stats = get_stats(ids)
        if not stats:
            break
        pair, freq = stats.most_common(1)[0]
        if freq < threshold:   # merge while freq >= threshold
            break
        # print(f"merging pair {pair} with frequency {freq} into new token {new_token}")
        ids = merge_once(ids, pair, new_token)
        bpe_map[new_token] = pair
        new_token += 1

    return ids, bpe_map

new_ids, bpe_map = train_bpe(tokens, start_token=256, threshold=10)


In [ ]:
print(f"len(original_ids):{len(tokens)}, len(new_ids):{len(new_ids)}", new_ids[: 1000])
print(f"len(bpe_map):{len(bpe_map)}", list(sorted(bpe_map.items(), reverse=True))[: 1000])

len(original_ids):250547, len(new_ids):71606 [333, 277, 688, 429, 325, 2645, 445, 169, 730, 295, 696, 731, 368, 1582, 184, 416, 277, 953, 184, 237, 157, 161, 429, 1819, 399, 145, 2646, 2647, 277, 696, 731, 401, 2648, 140, 1405, 278, 148, 576, 1484, 44, 961, 258, 1754, 2646, 2647, 277, 2812, 40, 531, 892, 284, 294, 41, 1761, 532, 1584, 136, 294, 1762, 2499, 2377, 333, 41, 10, 1243, 851, 1216, 10, 1820, 1536, 1217, 10, 2649, 2648, 140, 416, 236, 188, 128, 2500, 10, 553, 160, 917, 10, 1081, 644, 10, 763, 188, 513, 128, 10, 375, 409, 10, 118, 1149, 2499, 2503, 785, 2813, 333, 41, 324, 1821, 368, 483, 44, 1486, 44, 596, 1326, 44, 1131, 1150, 44, 937, 2142, 44, 2145, 334, 828, 644, 1822, 1639, 1700, 334, 1975, 2814, 697, 297, 1891, 2146, 32, 447, 1068, 297, 862, 447, 1821, 708, 917, 1765, 786, 1823, 1068, 509, 893, 160, 2504, 927, 880, 391, 491, 1892, 357, 1245, 2815, 295, 491, 1640, 839, 297, 1150, 534, 1010, 328, 146, 44, 1767, 44, 1822, 44, 1701, 346, 1586, 2378, 1641, 1273, 2505, 1132, 1

In [ ]:
def expand_bytes(token_id, bpe_map):
    if token_id < 256:
        return bytes([token_id])
    a, b = bpe_map[token_id]
    return expand_bytes(a, bpe_map) + expand_bytes(b, bpe_map)

def decode(token_ids, bpe_map):
    data = b''.join(expand_bytes(t, bpe_map) for t in token_ids)
    return data.decode('utf-8', errors='replace')

decode(new_ids, bpe_map)

'AI는 여기로 연결됩니다. 조류가 걸리는 호흡기 질병에 대해서는 조류 인플루엔자 문서를, 다른 뜻에 대해서는 AI (동음이의) 문서를 참고하십시오.\n시리즈의 일부\n인공지능 (AI)\n주요 목표\n접근 방식\n애플리케이션\n철학\n역사\n논란\n용어\nvte\n인공지능(영어: artificial intelligence, AI)은 컴퓨터가 학습, 추론, 지각, 언어 처리, 문제 해결, 계획과 의사 결정처럼 지능과 관련된 일을 수행하도록 하는 방법을 연구하는 컴퓨터 과학 분야이다. 이러한 방법으로 만든 프로그램이나 기계도 인공지능이라고 부른다. 인공지능 시스템은 입력을 처리해 예측값, 추천, 결정, 글·그림 같은 결과를 만들며, 반드시 사람의 사고 과정을 그대로 흉내 내는 것은 아니다.[1][2]\n인공지능에는 사람이 사실과 규칙을 직접 적어 넣는 기호주의 인공지능과, 여러 데이터에서 규칙성을 찾아 모델을 조정하는 기계 학습 등 다양한 접근이 있다. 딥 러닝은 여러 층의 인공 신경망을 사용하는 기계 학습의 한 갈래이고, 생성형 인공지능은 글·그림·음성·영상·코드 같은 새로운 결과물을 만든다. 인공지능은 검색·추천·번역·음성 및 영상 인식, 의료 지원, 게임과 로봇 제어 등에 쓰인다. 기계 학습과 생성형 인공지능은 인공지능 전체가 아니라 그 일부이다.[3][2]\n‘인공지능’이라는 이름은 1955년에 작성된 다트머스 여름 연구 계획서에서 사용되었고, 이듬해 열린 다트머스 회의는 관련 연구를 하나의 독립된 분야로 묶는 계기가 되었다. 이후 인공지능 연구의 중심은 논리와 규칙, 전문가 시스템, 통계적 기계 학습과 딥 러닝 등으로 넓어졌다. 2020년대에는 대규모 자료로 미리 학습한 파운데이션 모델과 생성형 인공지능이 확산되면서, 인공지능은 일상적인 정보 서비스뿐 아니라 과학, 의료, 교육, 산업과 창작 활동에도 널리 쓰이고 있다.[4][5]\n인공지능은 반복적인 일을 줄이고 정보 접근성을 높이며 과학·의료·교육에 새로운 도구를 제공할 수 있다. 한편 잘못된 결과, 

In [ ]:
def encode(text, bpe_map):
    tokens = list(map(int, text.encode('utf-8')))
    reversed_bpe_map = {v: k for k, v in bpe_map.items()}
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: reversed_bpe_map.get(p, float('inf')))
        if pair not in reversed_bpe_map:
            break
        idx = reversed_bpe_map[pair]
        tokens = merge_once(tokens, pair, idx)
    return tokens

print(encode("hello world!", bpe_map))

[104, 474, 311, 583, 2892, 33]


In [ ]:
print(decode(encode("AI는 여기로 연결됩니다. ", bpe_map), bpe_map))

AI는 여기로 연결됩니다. 


In [ ]:
import regex as re

gpt2 = re.compile(r"""'s|'t|'re|'ve|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2, "Hello world how are you"))

['Hello', ' world', ' how', ' are', ' you']


In [ ]:
import tiktoken

# GPT-2 (does not merge spaces)
# enc = tiktoken.get_encoding("gpt2")
# print(enc.encode("hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode(" hello world!!!"))

[24748, 1917, 12340]
